# Méthode de classification par espace latent

In [10]:
import torch 
import torch.nn as nn
import numpy as np
from torchvision import models

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F

DIM_IMG = 1800

class ArtVAE(nn.Module):
    def __init__(self, latent_dim=64):
        super(ArtVAE, self).__init__()
        self.latent_dim = latent_dim

        #Apprentissage de l'encoder 
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1),  # -> 32 x 64 x 64
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1), # -> 64 x 32 x 32
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),# -> 128 x 16 x 16
            nn.ReLU(),
            nn.Flatten()                                           
        )
        
        # Projections vers l'espace latent 
        self.fc_mu = nn.Linear(32768, latent_dim)
        self.fc_logvar = nn.Linear(32768, latent_dim)

        # Decoder
        self.decoder_input = nn.Linear(latent_dim, 32768)
        
        self.decoder = nn.Sequential(
            nn.Unflatten(1, (128, 16, 16)),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1), # -> 64 x 32 x 32
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),  # -> 32 x 64 x 64
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1),   # -> 3 x 128 x 128
            nn.Sigmoid() 
        )

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        # Encodage
        hidden = self.encoder(x)
        mu = self.fc_mu(hidden)
        logvar = self.fc_logvar(hidden)
        
        # Échantillonnage dans l'espace latent
        z = self.reparameterize(mu, logvar)
        
        # Décodage
        reconstruction = self.decoder(self.decoder_input(z))
        return reconstruction, mu, logvar

In [8]:
def vae_loss_function(recon_x, x, mu, logvar):
    bce = F.binary_cross_entropy(recon_x, x, reduction='sum')
    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return bce + kld

## Extraction de l'espace latent de notre VAE

In [11]:
def extract_latent_space(model, dataloader, device):
    model.eval()
    all_vectors = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            
            # On passe dans l'encodeur
            hidden = model.encoder(images)
            mu = model.fc_mu(hidden) # On utilise généralement 'mu' comme vecteur représentatif
            
            all_vectors.append(mu.cpu().numpy())
            all_labels.append(labels.numpy())
            
    return np.concatenate(all_vectors, axis=0), np.concatenate(all_labels, axis=0)

